# Explore sentiment-position directions and causal results

This read-only Colab notebook explores a completed sentiment-position run stored on Google Drive. It recreates Tigges **Figure 4-style** best-across-layer tables, cleaner cosine-similarity figures, and model-by-position logit-difference layer plots.

> Naming note: the best-across-layer direction comparison is Figure 4 in Tigges et al., not Table 1. Their actual Table 1 concerns summary-position patching. GPT-2 Small and Qwen3-0.6B are extensions of the Figure 4 presentation.

## Read-only contract

The notebook mounts Drive and reads existing CSV files. It does **not** create an analysis directory, modify the selected run, export tables, or save figures. All tables and plots are displayed only in notebook memory. No GPU, language-model loading, or Hugging Face token is required.

The completed run contains causal logit-difference and logit-flip metrics for Toy adjectives, Toy verbs, Toy adverbs, and SST. `toy_train` prompts and pairs were saved, but causal training-set patching metrics were not computed, so the Figure 4-style tables are evaluation-only.

## 1. Settings

In [ ]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional commit or tag for reproducible reporting code.
DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"

# None selects the newest completed run. Set an exact directory name to inspect another run.
RUN_ID = None  # Example: "2026-09-05_23-18_CDT"

## 2. Install the reporting package

The repository root is added explicitly because the package lives at `PROJECT_ROOT/sentiment_geometry`; this repository does not use a `src/` directory.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
importlib.invalidate_caches()

reporting = importlib.import_module("sentiment_geometry.reporting")
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(reporting.__file__).resolve().parents[1]
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )
required_apis = {
    "SentimentPositionReportData",
    "figure4_style_table",
    "plot_cosine_similarity_grid",
    "plot_cross_position_cosines",
    "plot_logit_difference_grid",
}
missing_apis = sorted(name for name in required_apis if not hasattr(reporting, name))
if missing_apis:
    raise ImportError(
        f"Reporting package is missing {missing_apis}. "
        "The checkout is older than this notebook; use a fresh runtime."
    )
print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)

## 3. Mount Drive and select a completed run

In [ ]:
import json

import pandas as pd
from google.colab import drive
from IPython.display import display

DRIVE_MOUNT_ROOT = Path("/content/drive")
if not (DRIVE_MOUNT_ROOT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)
else:
    print(f"Google Drive is already mounted at {DRIVE_MOUNT_ROOT}")

RUNS_ROOT = (
    Path(DRIVE_STORAGE_ROOT) / "sentiment-position-comparison" / "runs"
)
if not RUNS_ROOT.is_dir():
    raise FileNotFoundError(f"No sentiment-position runs found at {RUNS_ROOT}")

run_records = []
for run_directory in sorted(RUNS_ROOT.iterdir(), reverse=True):
    manifest_path = run_directory / "run_manifest.json"
    if not run_directory.is_dir() or not manifest_path.is_file():
        continue
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    run_records.append(
        {
            "run_id": run_directory.name,
            "status": manifest.get("status"),
            "started_at": manifest.get("started_at"),
            "completed_at": manifest.get("completed_at"),
            "direction_artifacts": manifest.get("direction_artifacts"),
        }
    )
runs = pd.DataFrame(run_records)
if runs.empty:
    raise RuntimeError(f"No readable run manifests found at {RUNS_ROOT}")
display(runs)

if RUN_ID is None:
    completed_runs = runs[runs["status"] == "completed"]
    if completed_runs.empty:
        raise RuntimeError("No completed runs are available. Set RUN_ID explicitly if needed.")
    selected_run_id = str(completed_runs.iloc[0]["run_id"])
else:
    if Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
        raise ValueError("RUN_ID must be one run-directory name, not a path.")
    selected_run_id = RUN_ID

RUN_ROOT = RUNS_ROOT / selected_run_id
RESULTS_ROOT = RUN_ROOT / "results"
if not RESULTS_ROOT.is_dir():
    raise FileNotFoundError(RESULTS_ROOT)
print("Selected read-only run:", RUN_ROOT)

## 4. Load and validate the saved result tables

In [ ]:
from sentiment_geometry.reporting import (
    DATASET_ORDER,
    MODEL_ORDER,
    SentimentPositionReportData,
    figure4_style_table,
    plot_cosine_similarity_grid,
    plot_cross_position_cosines,
    plot_logit_difference_grid,
)

report = SentimentPositionReportData.load(RESULTS_ROOT)
if report.has_training_causal_metrics:
    raise RuntimeError("Unexpected toy_train causal metrics were found; review run provenance.")
if report.evaluation_datasets != DATASET_ORDER:
    raise RuntimeError(
        f"Expected evaluation panels {DATASET_ORDER}; got {report.evaluation_datasets}"
    )
display(report.dataset_summary.reset_index(drop=True))
print(f"Loaded {len(report.metrics):,} aggregate causal-metric rows.")
print(f"Loaded {len(report.best_layers):,} best-layer rows.")
print(f"Loaded {len(report.direction_similarities):,} cosine-similarity rows.")
print("Training causal metrics available: no")

## 5. Tigges Figure 4-style best-across-layer tables

There are four tables: one for each model and fitting position. Every cell contains the independently best percentage across layers and the selected residual boundary. These are descriptive maxima, not one frozen layer reused across datasets or metrics.

In [ ]:
from IPython.display import Markdown

model_titles = {"gpt2-small": "GPT-2 Small", "qwen-0.6b": "Qwen3-0.6B Base"}
position_titles = {"adjective": "Adjective position", "final": "Final prompt-token position"}

for model in MODEL_ORDER:
    for position in ("adjective", "final"):
        display(Markdown(f"### {model_titles[model]} — {position_titles[position]}"))
        table = figure4_style_table(
            report.best_layers, model=model, fit_position=position
        )
        display(
            table.style
            .set_properties(**{"text-align": "center", "white-space": "pre-line"})
            .set_table_styles(
                [
                    {"selector": "th", "props": [("text-align", "center")]},
                    {"selector": "caption", "props": [("font-weight", "bold")]},
                ]
            )
        )

## 6. Cosine similarity among fitting methods

For each model, rows separate adjective and final-token fitting positions; columns show the first, middle, and last saved boundaries. Absolute cosine follows the paper's direction-line comparison convention.

In [ ]:
import matplotlib.pyplot as plt

for model in MODEL_ORDER:
    figure = plot_cosine_similarity_grid(report.direction_similarities, model=model)
    plt.show()
    plt.close(figure)

## 7. Adjective-versus-final direction similarity

These compact plots directly compare the two fitting positions for each method at the first, middle, and last boundaries.

In [ ]:
for model in MODEL_ORDER:
    figure = plot_cross_position_cosines(report.direction_similarities, model=model)
    plt.show()
    plt.close(figure)

## 8. Logit-difference percentage versus layer

One figure is displayed for each evaluation dataset. Every figure contains four subplots: model rows and fitting-position columns. The three fitting methods are the colored curves, and stars identify each curve's descriptive maximum. The dashed line marks 100% recovery; overshoot is deliberately not clipped.

In [ ]:
for dataset in report.evaluation_datasets:
    figure = plot_logit_difference_grid(report.metrics, dataset=dataset)
    plt.show()
    plt.close(figure)

## 9. Optional interactive filtering

Change the four selectors and rerun the next cell to inspect the exact saved layer rows behind any curve.

In [ ]:
SELECTED_MODEL = "gpt2-small"
SELECTED_POSITION = "adjective"
SELECTED_DATASET = "toy_adjectives"
SELECTED_METHOD = "das"

selection = report.metrics[
    (report.metrics["model"] == SELECTED_MODEL)
    & (report.metrics["fit_position"] == SELECTED_POSITION)
    & (report.metrics["dataset"] == SELECTED_DATASET)
    & (report.metrics["method"] == SELECTED_METHOD)
][
    [
        "layer",
        "n_directed_cases",
        "logit_difference_percent",
        "logit_flip_percent",
        "clean_accuracy",
        "patched_accuracy",
    ]
].sort_values("layer")
display(selection.reset_index(drop=True))

## Finished

No tables, figures, or other analysis artifacts were written to Google Drive. To discard all in-memory DataFrames and figures, restart the Colab runtime.